In [1]:
import torch
import torch.optim as optim
from torch.utils.data import DataLoader
from graphdataset import GraphDataset

from models import (
    ReLUFeedForwardNN,
    DropoutFeedForwardNN,
    ThreeLayerDropoutNN,
    BatchNormFeedForwardNN,
    SimpleCNN,
    SimpleRNN,
    TransformerClassifier,
    LSTM,
    ResNet,
)

from graphdataset import GraphDataset

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [2]:
# dataset stuff

TRAFFIC_FILE = "./data2/hops_12_nmap1_20000.csv"

dataset = GraphDataset(TRAFFIC_FILE)

# Split the dataset into training and testing sets
train_size = int(0.8 * len(dataset))
test_size = len(dataset) - train_size
train_dataset, test_dataset = torch.utils.data.random_split(
    dataset, [train_size, test_size]
)

# Define the data loaders
batch_size = 16
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)


parsing: 0/19000
parsing: 1000/19000
parsing: 2000/19000
parsing: 3000/19000
parsing: 4000/19000
parsing: 5000/19000
parsing: 6000/19000
parsing: 7000/19000
parsing: 8000/19000
parsing: 9000/19000
parsing: 10000/19000
parsing: 11000/19000
parsing: 12000/19000
parsing: 13000/19000
parsing: 14000/19000
parsing: 15000/19000
parsing: 16000/19000
parsing: 17000/19000
parsing: 18000/19000


In [3]:
dataset.features_df

,timeframe,n_unique_ips,n_unique_dst_ips,n_unique_dst_ports,n_accepted,n_dropped,n_connections,n_public,n_private,n_tcp,...,rolling_dropped,rolling_accepted,last_intersect_node_count,last_intersect_edge_count,degree_assortivity_coefficient,bridge_count,is_chordal,triangle_count,is_connected,label
0,14.0,210.0,103.0,28.0,176.0,274.0,450.0,209.0,1.0,234.0,...,0.000000,0.000000,140.0,66.0,-0.416517,151.0,0.0,210.0,0.0,0.0
1,15.0,211.0,102.0,29.0,238.0,207.0,445.0,210.0,1.0,234.0,...,0.000000,0.000000,137.0,62.0,-0.486230,153.0,0.0,211.0,0.0,0.0
2,16.0,209.0,110.0,28.0,288.0,210.0,498.0,208.0,1.0,319.0,...,0.000000,0.000000,144.0,73.0,-0.424269,144.0,0.0,209.0,0.0,0.0
3,17.0,259.0,163.0,26.0,597.0,192.0,789.0,258.0,1.0,542.0,...,0.000000,0.000000,148.0,78.0,-0.222291,203.0,0.0,259.0,0.0,0.0
4,18.0,215.0,112.0,31.0,476.0,245.0,721.0,214.0,1.0,491.0,...,0.000000,0.000000,138.0,71.0,-0.348465,148.0,0.0,215.0,0.0,0.0
5,19.0,248.0,151.0,30.0,311.0,203.0,514.0,247.0,1.0,267.0,...,0.000000,0.000000,140.0,70.0,-0.214594,185.0,0.0,248.0,0.0,0.0
6,20.0,260.0,155.0,32.0,379.0,256.0,635.0,259.0,1.0,445.0,...,0.000000,0.000000,153.0,76.0,-0.290769,191.0,0.0,260.0,0.0,0.0
7,21.0,207.0,123.0,30.0,377.0,182.0,559.0,206.0,1.0,362.0,...,0.000000,0.000000,148.0,85.0,-0.367382,150.0,0.0,207.0,0.0,0.0
8,22.0,271.0,171.0,34.0,521.0,279.0,800.0,270.0,1.0,541.0,...,0.000000,0.000000,149.0,83.0,-0.216859,208.0,0.0,271.0,0.0,0.0
9,23.0,219.0,119.0,28.0,489.0,205.0,694.0,218.0,1.0,174.0,...,0.000000,0.000000,163.0,91.0,-0.406255,157.0,0.0,219.0,0.0,0.0


In [4]:
models = [
    ReLUFeedForwardNN,
    DropoutFeedForwardNN,
    ThreeLayerDropoutNN,
    BatchNormFeedForwardNN,
    #SimpleCNN,
    SimpleRNN,
    TransformerClassifier,
    LSTM,
    ResNet,
]

for model in models:
    print(f' [i] Training using {model}')
    # Instantiate the model
    input_size = len(dataset.features_df_columns) - 1  # Exclude the label column
    hidden_size = 64
    output_size = 1  # Binary classification

    model = model(input_size, hidden_size, output_size).to(device)

    # Define the loss function and optimizer
    criterion = torch.nn.BCEWithLogitsLoss()
    optimizer = optim.Adam(model.parameters(), lr=0.01)

    num_epochs = 10

    # Training loop
    for epoch in range(num_epochs):
        running_loss = 0.0
        for i, data in enumerate(train_loader, 0):
            inputs, labels = data
            inputs, labels = inputs.to(device), labels.to(device)

            optimizer.zero_grad()

            outputs = model(inputs.float())
            #print(f"{outputs.shape=}")
            #print(f"{labels.float().shape=}")
            #print(f"{outputs=}")
            #print(f"{labels.float()}")
            loss = criterion(outputs, labels.float())
            loss.backward()
            optimizer.step()

            running_loss += loss.item()

        print(f"     Epoch: {epoch + 1}, Loss: {running_loss / len(train_loader)}")



 [i] Training using <class 'models.ReLUFeedForwardNN'>
     Epoch: 1, Loss: 29.90423583984375
     Epoch: 2, Loss: 11.86806869506836
     Epoch: 3, Loss: 2.701445370912552
     Epoch: 4, Loss: 3.731039822101593
     Epoch: 5, Loss: 1.2022705501804012
     Epoch: 6, Loss: 0.4830483943223953
     Epoch: 7, Loss: 1.5483834594488144
     Epoch: 8, Loss: 0.514153003692627
     Epoch: 9, Loss: 0.1036207489669323
     Epoch: 10, Loss: 0.27053875057026744
 [i] Training using <class 'models.DropoutFeedForwardNN'>
     Epoch: 1, Loss: 31.11566162109375
     Epoch: 2, Loss: 1.0604292154312134
     Epoch: 3, Loss: 0.6931471824645996
     Epoch: 4, Loss: 0.6931471824645996
     Epoch: 5, Loss: 0.6931471824645996
     Epoch: 6, Loss: 0.6931471824645996
     Epoch: 7, Loss: 0.6931471824645996
     Epoch: 8, Loss: 0.6931471824645996
     Epoch: 9, Loss: 0.6931471824645996
     Epoch: 10, Loss: 0.6931471824645996
 [i] Training using <class 'models.ThreeLayerDropoutNN'>
     Epoch: 1, Loss: 13.312129020